<a href="https://colab.research.google.com/github/purnabandekar/TNS/blob/main/Association_Rules_Apriori_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Association Rule Mining — the Apriori Algorithm

### Python Course · Day 15 · Colab Notebook

---

Last session we grouped **customers** who resembled each other. Today we are not looking at
the shoppers at all — we are looking at the **shopping**.

The question is: **which items keep turning up in the same basket?**

We will work out every number by hand first, on 20 baskets small enough to check with your
own eyes, and only then hand the job to a library.

---

**How to use this notebook**

1. Run every cell in order, top to bottom. Click a cell and press **Shift + Enter**.
2. **Every line of code has a comment** explaining what it does.
3. Most sections end with a **YOUR TURN** box — try it before moving on.
4. **Section 12 is a set of exercises with worked answers.**

**Save your own copy first:** File → Save a copy in Drive.

---

### What we are covering

| Section | Topic |
|---|---|
| 1 | The 20 baskets |
| 2 | Support — how common is it? |
| 3 | The cut-off, and round 1 of Apriori |
| 4 | Confidence — how reliable is it? |
| 5 | Lift — is the rule real? |
| 6 | Round 2: the pairs |
| 7 | Round 3: the pruning trick |
| 8 | Now the library |
| 9 | Generating the rules |
| 10 | Sorting: confidence vs lift |
| 11 | Changing the thresholds |
| 12 | **Exercises**, with answers |

---
## Section 1 — The 20 baskets

Each basket is one visit to the till. We store them as a **list of lists**: the outer list
holds the baskets, and each inner list holds the items in that basket.

Notice what is *not* here: no customer names, no prices, no dates, and **no answer column**.
Just what was in each basket.

In [ ]:
# a list of 20 baskets, each one a list of the items it contained
baskets = [
    ["bread", "butter", "milk"],
    ["bread", "butter", "milk", "eggs"],
    ["bread", "butter", "milk"],
    ["bread", "butter", "jam"],
    ["bread", "butter", "jam", "milk"],
    ["bread", "butter", "milk"],
    ["bread", "butter", "eggs"],
    ["bread", "butter", "milk", "jam"],
    ["bread", "butter", "milk"],
    ["bread", "milk", "jam"],
    ["bread", "milk", "eggs"],
    ["bread", "jam", "milk"],
    ["butter", "milk"],
    ["milk", "eggs"],
    ["milk", "eggs", "biscuits"],
    ["milk", "tea"],
    ["milk", "biscuits"],
    ["milk", "eggs"],
    ["eggs", "tea"],
    ["tea", "rice"],
]

# how many baskets we have in total - we will divide by this number constantly
n_baskets = len(baskets)
# print it
print("number of baskets:", n_baskets)

In [ ]:
# print the first five baskets, one per line, so we can see the shape of the data
for i in range(5):
    # i + 1 turns the position 0, 1, 2... into the basket names T1, T2, T3...
    print("T" + str(i + 1), ":", baskets[i])

We also want a list of every different item the shop sold, with no repeats.

In [ ]:
# start with an empty list
items = []
# look at every basket in turn
for basket in baskets:
    # and every item inside that basket
    for item in basket:
        # only add the item if we have not already got it
        if item not in items:
            # add it to our list of different items
            items.append(item)

# sort the list alphabetically so it always prints in the same order
items.sort()
# print how many different items there are, and what they are
print("different items:", len(items))
# print the items themselves
print(items)

> ### YOUR TURN
> Scroll up to the cell where the 20 baskets are written out and skim the whole list — the
> printout above shows only the first five. Which item do you think appears in the most
> baskets? Write your guess down; Section 3 will settle it.

---
## Section 2 — Support: how common is it?

**Support** is the simplest number in this session:

$$\text{support}(\{A, B\}) = \frac{\text{baskets containing both A and B}}{\text{all baskets}}$$

We need a way to count baskets containing a given set of items, so let us write one.

In [ ]:
# a small function: given a list of items, count how many baskets contain ALL of them
def count_baskets(wanted):
    # start the counter at zero
    total = 0
    # look at each basket in turn
    for basket in baskets:
        # assume this basket qualifies until we find a missing item
        has_all = True
        # check each item we are looking for
        for item in wanted:
            # if this basket is missing even one of them, it does not qualify
            if item not in basket:
                # one missing item is enough to disqualify this basket
                has_all = False
        # if nothing was missing, add this basket to the count
        if has_all:
            # add one to the count
            total = total + 1
    # hand the answer back
    return total


# a second function that turns that count into a support, by dividing by the total
def support(wanted):
    # the count of matching baskets, divided by how many baskets there are altogether
    return count_baskets(wanted) / n_baskets

In [ ]:
# how many baskets contain bread
print("baskets with bread        :", count_baskets(["bread"]))
# how many contain butter
print("baskets with butter       :", count_baskets(["butter"]))
# how many contain BOTH bread and butter
print("baskets with bread+butter :", count_baskets(["bread", "butter"]))

In [ ]:
# turn those counts into supports, rounded to 2 decimal places
print("support of bread        :", round(support(["bread"]), 2))
# the same for butter
print("support of butter       :", round(support(["butter"]), 2))
# and for the two of them together
print("support of bread+butter :", round(support(["bread", "butter"]), 2))

Read the last one out loud: **"bread and butter appear together in 45% of all baskets."**

Support does not care which item came first, or which one is more interesting. It is one
plain count.

> ### YOUR TURN
> Count how many of the 20 baskets contain milk — use the full list in the cell where the
> baskets are written out, not the five printed above. Divide by 20, then check yourself with
> `support(["milk"])`.

---
## Section 3 — The cut-off, and round 1 of Apriori

An itemset found in 1 basket out of 20 is not a pattern, it is a coincidence. So we set a
**minimum support** and throw away everything below it.

We will use **0.20**, which on 20 baskets means *"appears in at least 4 of them"*.

There is no correct value. Set it too high and you miss the rare-but-valuable pairings; set
it too low and you drown. **It is a judgement, and it is yours.**

In [ ]:
# our cut-off: an itemset must appear in at least this fraction of baskets to survive
min_support = 0.20

# print what that means in plain baskets
# int() drops the decimal point, so it reads as a whole number of baskets
print("min_support:", min_support, "which is", int(min_support * n_baskets), "baskets out of", n_baskets)

In [ ]:
# ROUND 1 of Apriori: count every single item, one at a time

# a list to collect the items that survive
frequent_items = []

# check each of our 8 items
for item in items:
    # work out its support
    s = support([item])
    # decide whether it is frequent enough to keep
    if s >= min_support:
        # keep it, and say so
        frequent_items.append(item)
        # ljust(9) pads the name so the columns line up
        print("KEEP  ", item.ljust(9), count_baskets([item]), "baskets  support", round(s, 2))
    # otherwise drop it
    else:
        # same line, with a different word at the front
        print("drop  ", item.ljust(9), count_baskets([item]), "baskets  support", round(s, 2))

# print the survivors
print()
# report how many we counted and how many survived
print("counted", len(items), "items, kept", len(frequent_items), ":", frequent_items)

Three items are gone already — and that matters more than it looks. Everything we do from
here only ever uses these **5 survivors**, so tea, biscuits and rice can never appear in any
pair or triple we examine.

> ### YOUR TURN
> `min_support` is set two cells above this one. Change it to `0.30`, then re-run **both**
> that cell and the round-1 cell, in that order. Which item drops out next?
>
> **Then set it back to 0.20 and re-run both cells again.** `min_support` and
> `frequent_items` are used all the way down to Section 10, so a stale value quietly changes
> every number below without ever showing an error.

---
## Section 4 — Confidence: how reliable is it?

Support treats a pair as one lump. **Confidence** splits it and gives the rule a direction:

$$\text{confidence}(A \rightarrow B) = \frac{\text{baskets with both}}{\text{baskets with A}}$$

Read it as a bet: *"if I see A in the basket, how often am I right that B is there too?"*

In [ ]:
# a function that works out the confidence of the rule "left -> right"
def confidence(left, right):
    # the baskets containing everything on the left AND everything on the right
    both = count_baskets(left + right)
    # the baskets containing just the left-hand side
    just_left = count_baskets(left)
    # divide one by the other
    return both / just_left


# the confidence that a bread basket also has butter
print("bread -> butter :", round(confidence(["bread"], ["butter"]), 4))
# the same pair, the other way round
print("butter -> bread :", round(confidence(["butter"], ["bread"]), 4))

**The two numbers are different, and that is the whole point of the arrow.**

- 9 of the 12 bread baskets have butter → 0.75
- 9 of the 10 butter baskets have bread → 0.90

Butter is the stronger signal here: butter almost never appears without bread, but bread
appears without butter fairly often.

In [ ]:
# show the counts behind those two numbers, so nothing is mysterious
print("baskets with bread          :", count_baskets(["bread"]))
# how many contain butter
print("baskets with butter         :", count_baskets(["butter"]))
# how many contain both
print("baskets with both           :", count_baskets(["bread", "butter"]))

> ### YOUR TURN
> Work out `confidence(["jam"], ["bread"])`. It comes out at exactly 1.00 — every single
> jam basket also had bread. Does that mean buying jam **causes** people to buy bread?

---
## Section 5 — Lift: is the rule real?

Here is the trap that catches everybody.

Milk is in **16 of our 20 baskets**. So almost anything you pick will have milk alongside it
— not because the two are related, but because milk is everywhere.

**Lift** fixes this by asking: *is B more likely once I know about A, than it was to begin
with?*

$$\text{lift}(A \rightarrow B) = \frac{\text{confidence}(A \rightarrow B)}{\text{support}(B)}$$

- **lift > 1** — A and B genuinely go together.
- **lift = 1** — no connection at all. Knowing about A told you nothing.
- **lift < 1** — they actually avoid each other.

In [ ]:
# a function for the lift of the rule "left -> right"
def lift(left, right):
    # the confidence of the rule
    c = confidence(left, right)
    # how common the right-hand side is on its own
    base = support(right)
    # divide one by the other
    return c / base


# a helper that prints all three numbers for one rule, neatly lined up
def show(left, right):
    # build a readable name for the rule, like "bread -> butter"
    name = "+".join(left) + " -> " + "+".join(right)
    # print the name padded to a fixed width, then the three numbers
    print(name.ljust(24),
          "support", format(support(left + right), ".2f"),
          "  confidence", format(confidence(left, right), ".2f"),
          "  lift", format(lift(left, right), ".2f"))

In [ ]:
# the rule that looks strong AND is strong
show(["bread"], ["butter"])
# the trap: high confidence, but lift of exactly 1
show(["butter"], ["milk"])

Look at those two lines carefully.

`butter -> milk` has a **confidence of 0.80**, which sounds impressive. But milk is in 80% of
baskets anyway, so 80% is exactly what you would have guessed without looking at the butter.

**Lift = 0.80 / 0.80 = 1.00. The rule tells you nothing at all.**

In [ ]:
# check that claim directly: how common is milk, with no conditions?
print("support of milk on its own   :", round(support(["milk"]), 2))
# and how common is milk once we know there is butter?
print("confidence of butter -> milk :", round(confidence(["butter"], ["milk"]), 2))
# identical numbers mean the butter changed nothing

In [ ]:
# a few more rules, so the pattern is clear
show(["jam"], ["bread"])
# a rule with high confidence but a lift barely above 1
show(["bread"], ["milk"])
# and one whose lift is actually below 1
show(["eggs"], ["milk"])

`eggs -> milk` has a lift **below 1**: eggs baskets look slightly *less* likely to contain
milk than an average basket is.

Be careful how hard you lean on that. Eggs are in 7 baskets and milk in 16, so pure chance
predicts 7 x 16 / 20 = 5.6 baskets with both, and we observed 5. **That is half a basket out
of twenty** — far too small to call a real effect. What matters here is the direction of the
lesson, not the number: confidence alone would not even have raised the question.

> ### YOUR TURN
> `bread -> milk` has confidence 0.83 — higher than `bread -> butter` at 0.75. Which of the
> two rules would you actually act on, and why?

---
## Section 6 — Round 2: the pairs

Round 1 gave us 5 surviving items. Round 2 builds every **pair** from those 5, counts them,
and keeps the frequent ones.

`itertools.combinations` gives us all the pairs without repeats: it produces
`("bread", "butter")` but not `("butter", "bread")`, because for support they are the same
thing.

In [ ]:
# bring in the combinations tool
import itertools

# build every possible pair from the 5 survivors
# the 2 means "groups of two"; list() turns the result into a plain list we can count
candidate_pairs = list(itertools.combinations(frequent_items, 2))

# print how many pairs there are and what they look like
print("pairs to check:", len(candidate_pairs))
# print the pairs themselves
print(candidate_pairs)

In [ ]:
# a list to collect the pairs that survive
frequent_pairs = []

# check each candidate pair
for pair in candidate_pairs:
    # combinations gives us a tuple, and our functions want a list
    pair_list = list(pair)
    # work out its support
    s = support(pair_list)
    # keep it if it clears the cut-off
    if s >= min_support:
        # keep this pair
        frequent_pairs.append(pair)
        # "+".join glues the two item names together for printing
        print("KEEP  ", ("+".join(pair)).ljust(18), count_baskets(pair_list), "baskets  support", round(s, 2))
    # otherwise report that we are dropping it
    else:
        # same line, different word at the front
        print("drop  ", ("+".join(pair)).ljust(18), count_baskets(pair_list), "baskets  support", round(s, 2))

# print the survivors
print()
# report how many pairs we counted and how many survived
print("counted", len(candidate_pairs), "pairs, kept", len(frequent_pairs))

Four pairs are gone. Keep an eye on **butter+jam**, which appears in only 3 baskets — it is
about to do a lot of work for us.

> ### YOUR TURN
> We checked 10 pairs. If we had *not* dropped tea, biscuits and rice in round 1, how many
> pairs would there have been? (There is a formula, but you can also just count: with 8
> items, every item pairs with the 7 after it, then 6, then 5...)

---
## Section 7 — Round 3: the pruning trick

This is the idea the whole algorithm is named after.

> **The Apriori principle:** if an itemset is rare, then every larger set containing it is at
> least as rare.

That has to be true. A basket containing {butter, jam, milk} necessarily contains
{butter, jam} — so the bigger set can never appear in more baskets than the smaller one.

So: **if butter+jam did not survive, we do not need to count anything containing it.** Not
one basket.

In [ ]:
# turn our surviving pairs into a set, so we can look pairs up quickly
frequent_pair_set = set(frequent_pairs)

# every possible triple from the 5 surviving items
all_triples = list(itertools.combinations(frequent_items, 3))
# print how many there are
print("possible triples:", len(all_triples))

In [ ]:
# lists to collect the triples worth counting and the ones we can skip
candidate_triples = []
# and the ones we can skip entirely
pruned_triples = []

# look at each possible triple
for triple in all_triples:
    # a triple is only worth counting if ALL THREE of its pairs survived round 2
    all_pairs_ok = True
    # check each of the three pairs hiding inside this triple
    for pair in itertools.combinations(triple, 2):
        # if this pair was dropped, the whole triple can be dropped too
        if pair not in frequent_pair_set:
            # one dead pair is enough to kill the whole triple
            all_pairs_ok = False
    # sort the triple into one of the two lists
    if all_pairs_ok:
        # worth counting
        candidate_triples.append(triple)
    # otherwise it goes on the skip list
    else:
        # we will never look at a single basket for this one
        pruned_triples.append(triple)

# report the split
print("worth counting :", len(candidate_triples), candidate_triples)
# and how many we skipped
print("pruned outright:", len(pruned_triples))

In [ ]:
# show exactly which dead pair killed each pruned triple
for triple in pruned_triples:
    # collect the pairs inside this triple that did not survive round 2
    killers = []
    # check each of the three pairs inside this triple
    for pair in itertools.combinations(triple, 2):
        # if this pair was dropped in round 2, it is one of the killers
        if pair not in frequent_pair_set:
            # record it in a readable form
            killers.append("+".join(pair))
    # print the triple and its killers
    print(("+".join(triple)).ljust(24), "killed by", killers)

In [ ]:
# now count only the survivors - just 2 of the 10
frequent_triples = []
# look at each triple that survived the pruning
for triple in candidate_triples:
    # work out the support of this triple
    s = support(list(triple))
    # keep it if it clears the cut-off
    if s >= min_support:
        # keep it
        frequent_triples.append(triple)
    # print it either way
    print(("+".join(triple)).ljust(24), count_baskets(list(triple)), "baskets  support", round(s, 2))

# print how many survived
print()
# report how many triples survived
print("frequent triples:", len(frequent_triples))

### What we just saved

| Round | Itemsets counted | Kept |
|---|---|---|
| 1 — single items | 8 | 5 |
| 2 — pairs | 10 | 6 |
| 3 — triples | **2** (not 10) | 2 |
| | **20 in total** | |

With 8 items there are **255** possible itemsets. We counted **20** of them.

On 8 items that is a nice saving. On a supermarket's 10,000 products it is the difference
between "runs in a second" and "will not finish before the sun burns out".

> ### YOUR TURN
> Would a four-item set be worth checking? A four-item set contains four different triples,
> and all four would have to be frequent. We have exactly two frequent triples. What does
> that tell you about round 4 — and therefore about whether we can stop here?

Work that one out and you have the last piece: round 4 has no candidates at all, so we can
stop, **and be certain we have not missed a single frequent itemset anywhere.**

---
## Section 8 — Now the library

We have done it all by hand, so nothing in the library is a mystery any more. `mlxtend` does
the same three steps, faster and in fewer lines.

*If Colab says it has installed a new version and asks you to restart the runtime, do it,
then run the cells from the top again.*

In [ ]:
# make sure mlxtend is installed, and new enough
# older versions of the library want an extra argument we are not going to pass,
# so we ask for at least version 0.24 rather than just "any mlxtend"
# --upgrade matters: without it, pip leaves an old version alone
!pip install -q --upgrade "mlxtend>=0.24"

In [ ]:
# import pandas, which the library uses to hand results back
import pandas as pd
# the tool that turns our lists of items into a table of True and False
from mlxtend.preprocessing import TransactionEncoder
# the two algorithms themselves
from mlxtend.frequent_patterns import apriori, association_rules

### Step 1 — reshape the baskets

The library does not want lists of words. It wants a table with **one row per basket and one
column per item**, holding True or False.

In [ ]:
# create the encoder
encoder = TransactionEncoder()
# fit() learns the list of all items, transform() builds the True/False grid
grid = encoder.fit(baskets).transform(baskets)

# put that grid into a pandas table, using the item names as column headings
table = pd.DataFrame(grid, columns=encoder.columns_)

# show the first 5 rows
print(table.head())

Each row is a basket, each column an item, and `True` means "this basket had this item".
That is the same information as our list of lists, just written differently.

In [ ]:
# Step 2 - find the frequent itemsets, with the same cut-off we used by hand
# use_colnames=True asks for item names in the output instead of column numbers
frequent_sets = apriori(table, min_support=min_support, use_colnames=True)

# sort them so the most common come first, and print
print(frequent_sets.sort_values("support", ascending=False).to_string(index=False))
# print a blank line, then one itemset on its own, to show what is actually stored
print()
# a frozenset is just a set that cannot be changed afterwards
print("one itemset on its own:", frequent_sets["itemsets"][0])

**13 itemsets: our 5 items, our 6 pairs, and our 2 triples.** The library agrees with our
hand count exactly.

Depending on your version of pandas, the itemsets print either as `frozenset({milk, bread})`
or as just `(milk, bread)`. **Both mean the same thing**, and the cell below shows what is
really stored.

> ### YOUR TURN
> Find `{bread, butter, milk}` in that list. Does its support match the number we worked out
> by hand in Section 7?

---
## Section 9 — Generating the rules

A frequent itemset is not a rule yet — it has no arrow in it. `association_rules` splits each
itemset every possible way and keeps the splits that clear a confidence threshold.

In [ ]:
# Step 3 - turn the frequent itemsets into rules
# metric and min_threshold together say "only keep rules with confidence 0.70 or better"
rules = association_rules(frequent_sets, metric="confidence", min_threshold=0.70)

# how many rules cleared the bar
print("rules found:", len(rules))

In [ ]:
# the output has a lot of columns, most of which we do not need today
# keep the five that matter and put them in a smaller table
simple = rules[["antecedents", "consequents", "support", "confidence", "lift"]]

# print them, rounded, without the row numbers
print(simple.round(3).to_string(index=False))

**antecedents** is the left of the arrow, **consequents** is the right. Those are the proper
names, and now you know what they mean.

Notice `{bread, butter, milk}` has produced several rules. Same three items, same support of
0.35, but the confidence changes depending on which side of the arrow each item sits.

---
## Section 10 — Sorting: confidence vs lift

Here is where the session pays off. Sort the same rules two different ways and watch the
order change.

In [ ]:
# sort the rules by confidence, best first
by_confidence = simple.sort_values("confidence", ascending=False)
# print the top 6
print("TOP 6 BY CONFIDENCE")
# head(6) takes the first six rows of the sorted table
print(by_confidence.head(6).round(3).to_string(index=False))

In [ ]:
# sort the same rules by lift instead
by_lift = simple.sort_values("lift", ascending=False)
# print the top 6
print("TOP 6 BY LIFT")
# the same six-row printout, from the differently sorted table
print(by_lift.head(6).round(3).to_string(index=False))

Different lists. Rules involving milk crowd the top of the confidence ranking and slide down
the lift ranking, because milk was in 80% of baskets to begin with.

Now throw away everything that lift says is uninformative.

In [ ]:
# keep only the rules where knowing the left-hand side actually changes the odds
# 1.1 is a judgement call: it means "at least 10% more likely than usual"
worthwhile = simple[simple["lift"] > 1.1]

# how many survived
print("rules with lift above 1.1:", len(worthwhile), "out of", len(simple))
# a blank line, to separate the count from the table
print()
# print them, best lift first
print(worthwhile.sort_values("lift", ascending=False).round(3).to_string(index=False))

From 14 rules down to 8. Look at what disappeared: **every rule that merely pointed AT milk
is gone.** Not one survivor has milk on its own as the right-hand side.

Milk has not vanished entirely — it still appears on the left of some rules, and inside a
couple of the right-hand sides. But "this basket will probably contain milk" has been struck
out everywhere, which is exactly right. Milk being in 80% of baskets is a fact about the
shop, not a pattern worth acting on.

> ### YOUR TURN
> Every surviving rule now has bread, butter or jam as the thing being predicted — milk is
> only ever along for the ride. If you ran this shop, what would you actually *do* with that?
> Name one concrete action.

---
## Section 11 — Changing the thresholds

The two numbers you choose — `min_support` and the confidence threshold — decide how many
rules come out the other end. Watch how sensitive that is.

In [ ]:
# try several support cut-offs and see how many itemsets and rules survive each
for cutoff in [0.10, 0.15, 0.20, 0.30, 0.40]:
    # find the frequent itemsets at this cut-off
    sets_here = apriori(table, min_support=cutoff, use_colnames=True)
    # turn them into rules at a fixed confidence of 0.7
    rules_here = association_rules(sets_here, metric="confidence", min_threshold=0.70)
    # report both counts
    print("min_support", format(cutoff, ".2f"),
          "->", str(len(sets_here)).rjust(2), "itemsets,",
          str(len(rules_here)).rjust(2), "rules")

Read the numbers rather than trusting a slogan. Halving the cut-off from 0.20 to 0.10 takes
us from 13 itemsets to 24, and from 14 rules to 18 — noticeable, but hardly an avalanche.

**On 20 baskets and 8 items there simply is not room for an avalanche.** The explosion from
the slides needs thousands of products; here we can see the shape of the effect but not its
size. Going the other way is the more useful lesson: at 0.40 you are down to 4 rules, and
you may well have thrown away something you needed.

**There is no correct threshold.** Start high, look at what comes out, and lower it only if
the rules are too few to be useful.

> ### YOUR TURN
> At `min_support = 0.40` how many rules survive? Is a shop with only that handful of rules
> better off, or worse off, than one drowning in hundreds?

---
## Section 12 — Exercises

Try each one before opening the answer underneath it.

### Exercise 1 — check the library by hand

Two rules are tied for the highest lift. Take `jam -> bread` and verify all three of its
numbers using the functions we wrote in Sections 2 to 5.

In [ ]:
# our own hand-written functions, on the rule the library ranked first
show(["jam"], ["bread"])

# now pull the library's own row for that same rule
# the two conditions pick the row whose left side is jam and right side is bread
same_rule = simple[(simple["antecedents"] == {"jam"}) & (simple["consequents"] == {"bread"})]

# a blank line, then the library's version of the numbers
print()
# print it without the row number
print(same_rule.round(4).to_string(index=False))

**Answer.** `jam -> bread` has support 0.25, confidence 1.00 and lift 1.67, both by hand and
from the library.

Confidence 1.00 means **every** jam basket also had bread — all 5 of them. That is a strong
rule on a tiny amount of evidence, and both halves of that sentence matter. Five baskets is
not much to bet a shelf layout on.

### Exercise 2 — find a rule that is worthless

Without looking, predict which item will produce lots of high-confidence, low-lift rules.
Then check.

In [ ]:
# find every rule whose right-hand side is milk on its own
milk_rules = simple[simple["consequents"] == {"milk"}]

# print them, highest confidence first
print(milk_rules.sort_values("confidence", ascending=False).round(3).to_string(index=False))

**Answer.** Milk, obviously — it is in 16 of the 20 baskets. Every rule pointing at milk has
a confidence between 0.71 and 0.83, which looks respectable, and a lift that never strays far
from 1.0 — one just above at 1.04, the rest sitting on it or below.

This is the single most useful habit from today: **when a rule points at your most popular
item, check the lift before you believe it.**

### Exercise 3 — the direction of the arrow

Compare `bread -> butter` with `butter -> bread`. They describe the same pair of items. Why
do they get different confidences but the same lift?

In [ ]:
# the same pair of items, written as a rule both ways round
show(["bread"], ["butter"])
# and now the other way round
show(["butter"], ["bread"])

**Answer.** Both rules count the same 9 baskets on top. What changes is the bottom:

- `bread -> butter` divides by the 12 bread baskets → 0.75
- `butter -> bread` divides by the 10 butter baskets → 0.90

Lift is the same **1.50** both ways, and that is not a coincidence — lift is
support(A and B) divided by support(A) times support(B), and swapping A and B leaves that
expression unchanged. **Confidence has a direction. Lift does not.**

### Exercise 4 — add a basket and watch it change

Add one more basket to the data and see how much the rules move.

In [ ]:
# make a copy of the baskets so the original is untouched
extra_baskets = list(baskets)
# add one more basket with jam but no bread - there is no such basket in the original 20
extra_baskets.append(["jam", "milk"])

# rebuild the True/False table for this longer list
extra_encoder = TransactionEncoder()
# fit and transform in one go, as before
extra_grid = extra_encoder.fit(extra_baskets).transform(extra_baskets)
# and use the encoder's own column names, so nothing can get out of step
extra_table = pd.DataFrame(extra_grid, columns=extra_encoder.columns_)

# run the whole pipeline again on the 21 baskets
extra_sets = apriori(extra_table, min_support=0.20, use_colnames=True)
# and turn them into rules with the same confidence threshold as before
extra_rules = association_rules(extra_sets, metric="confidence", min_threshold=0.70)

# find the jam -> bread rule in the new results
jam_rule = extra_rules[(extra_rules["antecedents"] == {"jam"}) &
                       (extra_rules["consequents"] == {"bread"})]
# print it, or say so if it did not survive
if len(jam_rule) > 0:
    # print the five columns we care about
    print(jam_rule[["antecedents", "consequents", "support", "confidence", "lift"]].round(3).to_string(index=False))
# otherwise the rule fell below the threshold and is not in the table at all
else:
    # say so in plain words
    print("jam -> bread did not survive the confidence threshold any more")

**Answer.** One extra basket drops the confidence of `jam -> bread` from a perfect 1.00 to
5 out of 6, which is about 0.83, and its lift falls with it.

**A rule built on 5 baskets is one shopper away from looking completely different.** Always
look at the support column before you get excited about a confidence of 1.00.

### Exercise 5 — the summary question

In two or three sentences, explain to somebody who missed the session what association rule
mining does, and the one mistake they are most likely to make with it.

**Answer.** A good answer says something like: *it looks through a pile of baskets and finds
items that turn up together, reporting them as IF-THEN rules with three numbers attached —
how common the pattern is, how reliable it is, and whether it is real. The mistake everybody
makes is trusting confidence on its own: a rule pointing at a very popular item will always
look confident, and only lift tells you whether knowing the left-hand side changed anything
at all.*

A really good answer adds: *and the arrow is not cause and effect.*

---
## Recap

| Number | Question it answers | How it is worked out |
|---|---|---|
| **Support** | is this combination common? | baskets with all the items ÷ all baskets |
| **Confidence** | if I see A, how often do I see B? | baskets with both ÷ baskets with A |
| **Lift** | did knowing about A change anything? | confidence ÷ support of B |

**Four habits worth keeping:**

1. **Sort by lift, not confidence.** A lift of 1.0 means the rule is telling you nothing,
   however confident it looks.
2. **Check the support column** before believing a confidence of 1.00 — it may rest on five
   baskets.
3. The Apriori principle — **rare set, rare supersets** — is what makes the search possible
   at all. Everything else is counting.
4. The arrow means *"these turn up together"*. It does **not** mean one causes the other.